# 01 — Spatial Alignment

Verify that all input layers register correctly to the base 1°×1° grid.

**Goal:** Confirm no systematic offsets or projection mismatches before feature engineering.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

## 1. Build Base Grid

In [ ]:
from preprocess.grid import build_base_grid, save_base_grid

grid = build_base_grid()
print(f"Base grid: {len(grid):,} cells")
print(f"Lon range: {grid['lon_centre'].min():.1f}° to {grid['lon_centre'].max():.1f}°")
print(f"Lat range: {grid['lat_centre'].min():.1f}° to {grid['lat_centre'].max():.1f}°")
display(grid.head())
save_base_grid(grid)
print("\n✅ Base grid saved.")

## 2. Assign Hotspots to Grid

In [ ]:
from ingest.hotspot_catalog import load_hotspot_catalog
from preprocess.align_layers import assign_hotspots_to_grid, save_hotspot_grid

catalog = load_hotspot_catalog()
labeled_grid = assign_hotspots_to_grid(grid, catalog)

n_pos = labeled_grid['has_hotspot'].sum()
print(f"Positive cells: {n_pos} / {len(labeled_grid):,} ({100*n_pos/len(labeled_grid):.2f}%)")
display(labeled_grid[labeled_grid['has_hotspot']==1].head(10))

In [ ]:
# Visual check: do hotspot-containing cells match catalog locations?
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Original catalog
axes[0].scatter(catalog['longitude'], catalog['latitude'], 
                s=10, alpha=0.7, color='orange')
axes[0].set_title('Catalog: raw locations')
axes[0].set_xlim(-180, 180); axes[0].set_ylim(-90, 90)
axes[0].grid(alpha=0.3)

# Grid-assigned
pos = labeled_grid[labeled_grid['has_hotspot']==1]
axes[1].scatter(pos['lon_centre'], pos['lat_centre'], 
                s=10, alpha=0.7, color='red')
axes[1].set_title('Grid: assigned cell centres (1°×1°)')
axes[1].set_xlim(-180, 180); axes[1].set_ylim(-90, 90)
axes[1].grid(alpha=0.3)

plt.suptitle('Alignment Check: Catalog vs Grid Assignment', fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✅ Visual inspection: distributions should match at ~1° resolution.")
print("   Any systematic offset would indicate a CRS or coordinate convention issue.")

## 3. Alignment Summary

- [ ] Base grid saved to `data/processed/`
- [ ] Hotspot grid cells visually match catalog locations
- [ ] No systematic spatial offset detected
- [ ] Cell count matches expectation (64,800 for 1°×1°)